In [ ]:
import polars as pl
from pathlib import Path
from sensession.campaign import CampaignProcessor
import hvplot.polars  # noqa: F401
from loguru import logger
import numpy as np
from copy import deepcopy

logger.remove()

In [ ]:
exp_dir = Path.cwd() / ".." / ".." / "data" / "doppler_emulation_slow"
data = pl.read_parquet(exp_dir / "csi.parquet")
meta = pl.read_parquet(exp_dir / "meta.parquet")

# Equalize data appropriately
proc = (
    CampaignProcessor(csi=data, meta=meta)
    .correct_rssi_by_agc()
    .unwrap()
    .filter("antenna_idxs", 0)
    .filter("receiver_name", "asus1")
    .filter("meta_id", "7a35e641b2eaa6a165308d9062cfe963")  # non warmup
)


proc_simple = deepcopy(proc)
proc_lsqfit = deepcopy(proc)

In [ ]:
def get_csi(df: pl.DataFrame) -> np.ndarray:
    return df.csi.drop(
        "stream_idxs",
        "antenna_idxs",
        "rssi",
        "antenna_rssi",
        "timestamp",
        "rx_antenna_capture_num",
        "collection_name",
    )


proc_simple = proc_simple.detrend_phase()

proc_lsqfit = proc_lsqfit.detrend_phase_ls()


csi_simple = get_csi(proc_simple)
csi_lsqfit = get_csi(proc_lsqfit)

In [ ]:
csi_simple.hvplot.scatter(
    x="subcarrier_idxs",
    y="csi_phase",
    groupby=["capture_num"],
    width=1200,
    ylim=(-1.5, 1.5),
)

In [ ]:
csi_lsqfit.hvplot.scatter(
    x="subcarrier_idxs",
    y="csi_phase",
    groupby=["capture_num"],
    width=1200,
    ylim=(-1.5, 1.5),
)